# register-back-fn-after-wrap — faded example 1: Register the Backward Function for torch.sqrt (Faded)

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `register-back-fn-after-wrap`. The last cell reports your progress on the `Backprop: register back fn` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: register back fn` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`register-back-fn-after-wrap`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "register-back-fn-after-wrap"
DD_SUBTOPIC = "Backprop: register back fn"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The derivative of `sqrt(x)` with respect to `x` is `1 / (2 * sqrt(x))`. When registering a back function for `torch.sqrt`, you store `(t.sqrt, 0)` in the table because `sqrt` is a unary operation. The back function receives `(grad_out, out, x)` and uses the saved output `out = sqrt(x)` to avoid recomputing the square root.

## Faded exercise 1

The `BackwardFuncLookup` class and `register_sqrt` function are partially provided. Your task is to **implement the body of `sqrt_back`** — the backward function for `torch.sqrt`.

Recall: if `out = sqrt(x)`, then `d(out)/d(x) = 1 / (2 * sqrt(x)) = 1 / (2 * out)`.

The back function signature is `sqrt_back(grad_out, out, x) -> Tensor`. Use `out` (the saved forward output) rather than recomputing the square root.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t

class BackwardFuncLookup:
    def __init__(self):
        self._table = {}
    def add_back_func(self, fwd_fn, argnum, back_fn):
        self._table[(fwd_fn, argnum)] = back_fn
    def get_back_func(self, fwd_fn, argnum):
        return self._table[(fwd_fn, argnum)]

def sqrt_back(grad_out: t.Tensor, out: t.Tensor, x: t.Tensor) -> t.Tensor:
    return grad_out / (2.0 * out)

def register_sqrt(BACK_FUNCS: BackwardFuncLookup) -> None:
    BACK_FUNCS.add_back_func(t.sqrt, 0, sqrt_back)


def _test():
    import torch as t
    x = t.tensor([1.0, 4.0, 9.0])
    out = t.sqrt(x)
    grad_out = t.ones_like(x)
    result = sqrt_back(grad_out, out, x)
    expected = grad_out / (2.0 * t.sqrt(x))
    assert t.allclose(result, expected, atol=1e-6), f'Expected {expected}, got {result}'
    # Also verify table lookup works
    BACK_FUNCS = BackwardFuncLookup()
    register_sqrt(BACK_FUNCS)
    fn = BACK_FUNCS.get_back_func(t.sqrt, 0)
    assert fn is sqrt_back, 'Expected sqrt_back to be registered at (t.sqrt, 0)'


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

class BackwardFuncLookup:
    def __init__(self):
        self._table = {}
    def add_back_func(self, fwd_fn, argnum, back_fn):
        self._table[(fwd_fn, argnum)] = back_fn
    def get_back_func(self, fwd_fn, argnum):
        return self._table[(fwd_fn, argnum)]

def sqrt_back(grad_out: t.Tensor, out: t.Tensor, x: t.Tensor) -> t.Tensor:
    return grad_out / (2.0 * out)

def register_sqrt(BACK_FUNCS: BackwardFuncLookup) -> None:
    BACK_FUNCS.add_back_func(t.sqrt, 0, sqrt_back)
```
</details>